# 06 Clustered Retrieval

Purpose: group similar jobs into role clusters and score degrees against those cluster centroids.

Inputs:
- `data/interim/jobs_clean.parquet`
- `data/processed/degree_modules.parquet`
- `data/processed/degree_profiles.parquet`
- `data/embeddings/degree_module_embeddings_all-MiniLM-L6-v2.npy`
- `data/embeddings/job_embeddings_all-MiniLM-L6-v2.npy`

Outputs:
- `data/processed/jobs_with_clusters.parquet`
- `data/processed/job_cluster_summary.csv`
- `data/embeddings/job_clusters_k75.npy`
- `data/embeddings/degree_cluster_similarity_k75.npy`
- `data/processed/degree_cluster_alignment.csv`


In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.io import paths, require_files

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

from analysis.clustering import build_degree_cluster_alignment, compute_degree_cluster_similarity, label_clusters, load_or_compute_clusters
from analysis.profiles import build_degree_module_indices

MODEL_NAME = 'all-MiniLM-L6-v2'
DEGREE_MODULE_TOP_K = 5
N_CLUSTERS = 75

require_files([
    paths.jobs_clean_parquet,
    paths.degree_modules_parquet,
    paths.degree_profiles_parquet,
    paths.degree_module_embeddings_npy(MODEL_NAME),
    paths.job_embeddings_npy(MODEL_NAME),
])
print('Expected upstream inputs: cleaned jobs plus baseline embeddings')


Expected upstream inputs: cleaned jobs plus baseline embeddings


In [2]:
jobs = pd.read_parquet(paths.jobs_clean_parquet)
degree_modules = pd.read_parquet(paths.degree_modules_parquet)
degree_profiles = pd.read_parquet(paths.degree_profiles_parquet)
module_embeddings = np.load(paths.degree_module_embeddings_npy(MODEL_NAME))
job_embeddings = np.load(paths.job_embeddings_npy(MODEL_NAME))
degree_module_indices = build_degree_module_indices(degree_profiles, degree_modules)

cluster_labels = load_or_compute_clusters(job_embeddings, paths.cluster_labels_npy(N_CLUSTERS), n_clusters=N_CLUSTERS)
jobs_with_clusters, cluster_df = label_clusters(jobs, cluster_labels, n_clusters=N_CLUSTERS)
jobs_with_clusters.to_parquet(paths.jobs_with_clusters_parquet, index=False)
cluster_df.to_csv(paths.cluster_summary_csv, index=False)

cluster_sim_path = paths.degree_cluster_similarity_npy(N_CLUSTERS)
if cluster_sim_path.exists():
    degree_cluster_sim = np.load(cluster_sim_path)
    expected_shape = (len(degree_profiles), N_CLUSTERS)
    if degree_cluster_sim.shape != expected_shape:
        degree_cluster_sim = compute_degree_cluster_similarity(
            degree_profiles, degree_module_indices, module_embeddings, job_embeddings, cluster_labels, DEGREE_MODULE_TOP_K, N_CLUSTERS
        )
        np.save(cluster_sim_path, degree_cluster_sim)
else:
    degree_cluster_sim = compute_degree_cluster_similarity(
        degree_profiles, degree_module_indices, module_embeddings, job_embeddings, cluster_labels, DEGREE_MODULE_TOP_K, N_CLUSTERS
    )
    np.save(cluster_sim_path, degree_cluster_sim)

alignment_table = build_degree_cluster_alignment(degree_profiles, degree_cluster_sim, cluster_df, top_n=5)
alignment_table.to_csv(paths.degree_cluster_alignment_csv, index=False)

print(f'Saved jobs with clusters to: {paths.jobs_with_clusters_parquet}')
print(f'Saved cluster summary to: {paths.cluster_summary_csv}')
print(f'Saved degree-cluster similarity to: {cluster_sim_path}')
print(f'Saved degree-cluster alignment to: {paths.degree_cluster_alignment_csv}')


Saved jobs with clusters to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/processed/jobs_with_clusters.parquet
Saved cluster summary to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/processed/job_cluster_summary.csv
Saved degree-cluster similarity to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/embeddings/degree_cluster_similarity_k75.npy
Saved degree-cluster alignment to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/processed/degree_cluster_alignment.csv


In [3]:
display(cluster_df.head(20))
print('Sample: top 3 cluster matches per degree')
display(alignment_table[alignment_table['rank'] <= 3].head(30))


,cluster_id,n_jobs,top_category,top_terms,common_titles
0,25,504,Unknown,"salary, sales, days, location, hours, working,...","|, -, to, sales, /"
1,49,481,Unknown,"sales, customer, retail, customers, store, pro...","sales, manager, executive, retail, assistant"
2,13,467,Unknown,"administrative, office, support, filing, tasks...","admin, assistant, executive, administrative, a..."
3,65,454,Unknown,"accounting, accounts, financial, tax, finance,...","accounts, executive, assistant, accountant, ac..."
4,17,424,Unknown,"food, kitchen, chef, hygiene, dishes, menu, co...","chef, cook, de, sous, assistant"
5,23,420,Unknown,"warehouse, inventory, logistics, goods, stock,...","warehouse, executive, logistics, assistant, ma..."
6,40,393,Unknown,"project, management, projects, site, construct...","project, manager, coordinator, /, executive"
7,2,375,Unknown,"sales, business, new, market, business develop...","executive, business, sales, development, manager"
8,28,349,Unknown,"project, site, engineering, design, constructi...","engineer, project, electrical, senior, design"
9,4,336,Unknown,"food, restaurant, service, staff, outlet, cust...","manager, supervisor, restaurant, assistant, f&b"


Sample: top 3 cluster matches per degree


,degree_id,degree,rank,cluster_id,cluster_label,cluster_terms,cluster_score
0,biz,Business Administration,1,32,Unknown,"financial, finance, accounting, reporting, tax...",0.5491
1,biz,Business Administration,2,7,Unknown,"sales, procurement, business, market, regional...",0.5489
2,biz,Business Administration,3,2,Unknown,"sales, business, new, market, business develop...",0.5375
5,ce,Civil Engineering,1,28,Unknown,"project, site, engineering, design, constructi...",0.5025
6,ce,Civil Engineering,2,12,Unknown,"construction, site, project, works, civil, str...",0.4822
7,ce,Civil Engineering,3,41,Unknown,"design, interior, project, interior design, cr...",0.4774
10,cnm,Communications and New Media,1,24,Unknown,"content, media, marketing, social, social medi...",0.5395
11,cnm,Communications and New Media,2,63,Unknown,"product, design, development, experience, busi...",0.4622
12,cnm,Communications and New Media,3,31,Unknown,"students, student, school, learning, teaching,...",0.4436
15,cs,Computer Science,1,43,Unknown,"software, development, experience, design, jav...",0.4449
